In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Introdução às primitivas

<Accordion>
<AccordionItem title="Package versions">

O código nesta página foi desenvolvido com os seguintes requisitos.
Recomendamos usar essas versões ou versões mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## Por que o Qiskit introduziu as primitivas?
De modo similar aos primórdios dos computadores clássicos, quando os desenvolvedores precisavam manipular registradores de CPU diretamente, a interface inicial com QPUs simplesmente retornava os dados brutos da eletrônica de controle.
Isso não era um grande problema quando as QPUs ficavam em laboratórios e só permitiam acesso direto por pesquisadores.
Reconhecendo que a maioria dos desenvolvedores não precisaria nem deveria se familiarizar com a conversão desses dados brutos em 0s e 1s, o Qiskit introduziu o `backend.run`, uma primeira abstração para acessar QPUs na nuvem. Isso permitiu que os desenvolvedores
operassem em um formato de dados familiar e se concentrassem no panorama geral.

À medida que o acesso às QPUs se tornou mais difundido, e com mais algoritmos quânticos sendo desenvolvidos,
surgiu novamente a necessidade de uma abstração de nível mais alto. Em resposta, o Qiskit introduziu
a interface de primitivas, que é otimizada para duas tarefas centrais no desenvolvimento de algoritmos quânticos:
estimativa de valor esperado (`Estimator`) e amostragem de circuitos (`Sampler`). O objetivo é, mais uma vez,
ajudar os desenvolvedores a se concentrarem mais na inovação e menos na conversão de dados. A interface de primitivas substitui a interface `backend.run`, já que o `Sampler` oferece o mesmo acesso direto ao hardware que era fornecido pelo `backend.run`.

## O que é uma primitiva?
Os sistemas computacionais são construídos em múltiplas camadas de abstração. As abstrações permitem que você se concentre em um nível específico de detalhe relevante para a tarefa em questão. Quanto mais próximo você está do hardware,
menor é o nível de abstração necessário (por exemplo, talvez você precise mover ou manipular dados no nível de instrução da CPU). Quanto mais complexa a tarefa que você deseja realizar,
mais alto será o nível das abstrações (por exemplo, você pode estar usando uma biblioteca de programação para realizar
cálculos algébricos).

Nesse contexto, uma *primitiva* é a menor instrução de processamento, o bloco de construção mais simples a partir do qual
é possível criar algo útil para um dado nível de abstração.

O progresso recente na computação quântica aumentou a necessidade de trabalhar em níveis mais altos de abstração.
À medida que o campo avança em direção a unidades de processamento quântico (QPUs) maiores e fluxos de trabalho mais complexos, o foco muda de interagir com sinais de qubits individuais para encarar os dispositivos quânticos como sistemas que executam as tarefas necessárias.

As duas tarefas mais comuns para computadores quânticos são a amostragem de estados quânticos e o cálculo de valores esperados.
Essas tarefas motivaram o design das primitivas do Qiskit: **Estimator** e **Sampler**.

- O Estimator calcula os valores esperados de observáveis em relação a estados preparados por circuitos quânticos.
- O Sampler amostra o registrador de saída da execução de circuitos quânticos.

Em resumo, o modelo computacional introduzido pelas primitivas do Qiskit aproxima a programação quântica um passo a mais
de onde a programação clássica está hoje, onde o foco está menos nos detalhes do hardware e mais nos resultados
que você está tentando alcançar.

## Definição e implementações das primitivas
Existem dois tipos de primitivas Qiskit: as classes base e suas implementações. As primitivas Estimator e Sampler são definidas por classes base de primitivas de código aberto que residem no SDK do Qiskit (no módulo [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Provedores (como o Qiskit Runtime) podem usar essas classes base para derivar suas próprias implementações de Sampler e Estimator. A maioria dos usuários interagirá com as implementações dos provedores, não com as primitivas base.

### Classes base
As primitivas `Base` são classes abstratas que definem uma interface comum para implementar primitivas. Todas as outras classes no módulo [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) herdam dessas classes base. Os desenvolvedores devem usá-las se tiverem interesse em criar seu próprio modelo de execução baseado em primitivas para um provedor específico. Essas classes também podem ser úteis para quem deseja fazer um processamento altamente personalizado e descobre que as implementações de primitivas existentes são simples demais para suas necessidades. Usuários em geral não utilizarão as classes base diretamente.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) e [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) — Embora as primitivas V1 ainda sejam utilizáveis, estes guias focam nas primitivas V2 por serem as mais recentes e as mais comumente usadas.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) e [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) — As primitivas de referência do Qiskit seguem essas especificações de interface.

<span id="implementations"></span>
### Implementações
Todas as primitivas são criadas a partir das classes base; portanto, têm a mesma estrutura e uso gerais. Por exemplo, o formato de entrada para todas as primitivas do Estimator é o mesmo. No entanto, há diferenças nas implementações que as tornam únicas.

Estas são implementações das classes base de primitivas:

- As [primitivas do Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) e [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), oferecem uma implementação mais sofisticada (por exemplo, incluindo mitigação de erros) como um serviço baseado em nuvem. Essa implementação das primitivas base é usada para acessar o hardware do IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) e [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) — Implementações de referência das primitivas que usam o simulador embutido no Qiskit. Elas são construídas com o módulo [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) do Qiskit, produzindo resultados baseados em simulações ideais de vetor de estado. São acessadas por meio do Qiskit. Consulte [Simulação exata com primitivas do Qiskit](/guides/simulate-with-qiskit-sdk-primitives) para detalhes de uso.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) e [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) — Você pode usar essas classes para "envolver" qualquer recurso de computação quântica em uma primitiva. Isso permite escrever código no estilo de primitivas para provedores que ainda não possuem uma interface baseada em primitivas. Essas classes podem ser usadas exatamente como o Sampler e o Estimator regulares, exceto que devem ser inicializadas com um argumento adicional `backend` para selecionar em qual computador quântico executar. São acessadas usando o Qiskit. Consulte o guia de [primitivas de backend](/guides/get-started-with-backend-primitives) para mais informações.

## Opções
Você pode passar opções às primitivas para personalizá-las de acordo com suas necessidades. Embora a interface do método `run()` das primitivas seja comum a todas as implementações, suas opções não são. Consulte as referências de API de uma implementação de primitiva específica para saber quais opções ela suporta.

Por exemplo, consulte os tópicos [Opções do Estimator](/guides/estimator-options) e [Opções do Sampler](/guides/sampler-options) para saber sobre as opções das primitivas do Qiskit Runtime, ou veja as [referências de API do Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) para as opções das primitivas do Qiskit Aer.

## Benefícios das primitivas do Qiskit
Com as primitivas, os usuários do Qiskit podem escrever código quântico para uma QPU específica sem precisar gerenciar explicitamente cada detalhe. Além disso, por causa da camada adicional de abstração, você pode ser capaz de acessar com mais facilidade
as capacidades avançadas de hardware de um determinado provedor. Por exemplo, com as primitivas do Qiskit Runtime,
você pode aproveitar os avanços mais recentes em mitigação e supressão de erros alternando opções como o [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) da primitiva, em vez de construir sua própria implementação dessas técnicas.

Para provedores de hardware, implementar primitivas nativamente significa que você pode oferecer aos seus usuários uma forma mais "pronta para uso" de acessar os recursos do seu hardware, como técnicas avançadas de pós-processamento. Portanto, fica mais fácil para seus usuários se beneficiarem das melhores capacidades do seu hardware.

## Próximos passos
> **Tip:** - Entenda a [entrada e saída de primitivas](/guides/primitive-input-output).
> - Revise [exemplos detalhados](/guides/simulate-with-qiskit-sdk-primitives).
> - Pratique com primitivas trabalhando na [lição de função de custo](/learning/courses/variational-algorithm-design/cost-functions) no IBM Quantum Learning.
> - Revise [Criar um provedor](/guides/create-a-provider) para aprender como implementar suas próprias primitivas Sampler e Estimator.
> - Consulte as [referências de API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - Leia [Migrar para primitivas V2](/guides/v2-primitives).
> - Saiba mais sobre as [primitivas do Qiskit Runtime](/guides/qiskit-runtime-primitives), que são usadas para executar circuitos em QPUs da IBM.